# Active-learning runs: Random, PPAL, Rarity-aware, Entropy-ENMS-DivProto

Runs the fixed protocol for the four strategies and the seeds in the config. Runs resume by round through `RESUME_DIR`.

In [ ]:
!pip install -q ultralytics==8.4.51 scipy==1.17.1 pyyaml==6.0.3

In [ ]:
# Set these three values to match the two Kaggle Datasets added as input.
CODE_DIR   = "/kaggle/input/albench-code"
EXPORT_DIR = "/kaggle/input/<data-slug>/export_sortwaste"
CONFIG     = "sortwaste"

RESUME_DIR      = ""     # output folder of a previous session, empty for a new run
ONLY_STRATEGIES = []     # for example ["rarity_aware", "entropy"], empty for all four
ONLY_SEEDS      = []     # for example [13], empty for the first al.seeds_n seeds
RUN_REPORT      = True
USE_ALL_GPUS    = True

In [ ]:
import json, os, shutil, sys
from pathlib import Path
import yaml

WORK = Path("/kaggle/working")
os.chdir(WORK)
CODE = Path(CODE_DIR)
EXPORT = Path(EXPORT_DIR).resolve()
assert (CODE / "albench").is_dir(), f"albench/ not found in {CODE}"
assert (CODE / "configs").is_dir(), f"configs/ not found in {CODE}"
assert EXPORT.is_dir(), f"EXPORT_DIR does not exist: {EXPORT}"
assert (CODE / "yolov8n.pt").is_file(), "yolov8n.pt not found in the code dataset"

os.environ["ALBENCH_WORK"] = str(WORK)
os.environ["DATA_ROOT"] = str(EXPORT.parent)
sys.path.insert(0, str(CODE))

import scipy, torch, ultralytics
assert ultralytics.__version__ == "8.4.51", ultralytics.__version__
assert scipy.__version__ == "1.17.1", scipy.__version__
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings."
N_GPU = torch.cuda.device_count() if USE_ALL_GPUS else 1
print("torch", torch.__version__, "CUDA", torch.version.cuda)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])

# Check the extracted tree against the manifest, so a missing ZIP part stops here.
manifest_path = EXPORT.parent / f"{CONFIG}-data-manifest.json"
assert manifest_path.is_file(), f"manifest not found: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["export_dir"] == EXPORT.name
expected = {item["path"]: int(item["bytes"]) for item in manifest["files"]}
actual = {p.relative_to(EXPORT.parent).as_posix(): p.stat().st_size
          for p in EXPORT.rglob("*") if p.is_file() and p.suffix != ".cache"}
missing = sorted(set(expected) - set(actual))
extra = sorted(set(actual) - set(expected))
wrong = sorted(k for k in set(expected) & set(actual) if expected[k] != actual[k])
assert not (missing or extra or wrong), (missing[:5], extra[:5], wrong[:5])
print("Dataset manifest OK:", len(actual), "files")

from albench.config import load_config, load_seeds
cfg = load_config(CONFIG)
cfg["baseline"]["cache"] = False
cfg["al"]["cache"] = False
cfg["baseline"]["model"] = cfg["al"]["model"] = str(CODE / "yolov8n.pt")
if ONLY_STRATEGIES:
    unknown = set(ONLY_STRATEGIES) - set(cfg["al"]["strategies"])
    assert not unknown, f"strategies not in the config: {sorted(unknown)}"
    cfg["al"]["strategies"] = ONLY_STRATEGIES
cfg["baseline"]["workers"] = 2
cfg["al"]["workers"] = 2
assert cfg["al"]["batch"] == 64, "the protocol fixes batch=64"
assert cfg["al"]["epochs"] == 100, "the protocol fixes 100 epochs"
assert cfg["al"]["warmup_epochs"] == 0.0, "the protocol fixes warmup_epochs=0"
assert cfg["al"]["close_mosaic"] == 0, "the protocol fixes close_mosaic=0"
assert cfg["al"]["eval_checkpoint"] == "last", "the protocol uses last.pt"

CFG_KAGGLE = WORK / "config_kaggle.yaml"
CFG_KAGGLE.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding="utf-8")
source_yaml = EXPORT / "data.yaml"
d = yaml.safe_load(source_yaml.read_text(encoding="utf-8"))
d.update(path=str(EXPORT), nc=cfg["classes"]["nc"], names=cfg["classes"]["names"])
DATA_YAML = WORK / "data_kaggle.yaml"
DATA_YAML.write_text(yaml.safe_dump(d, allow_unicode=True), encoding="utf-8")
for split, counts in manifest["split_counts"].items():
    print(split, counts)
print("config ->", CFG_KAGGLE)

In [ ]:
# Restore the state, splits, reports and runs of the previous session.
if RESUME_DIR:
    resume = Path(RESUME_DIR)
    for sub in ("state", "splits", "reports", "runs"):
        source = resume / sub
        if source.exists():
            shutil.copytree(source, WORK / sub, dirs_exist_ok=True)
            print("resumed", sub)
else:
    print("New run")

In [ ]:
import subprocess, time

# One process per GPU, split by seed. results.csv is protected by a file lock.
seeds = ONLY_SEEDS or load_seeds()[:cfg["al"]["seeds_n"]]
shards = [s for s in (seeds[i::N_GPU] for i in range(N_GPU)) if s]
print("strategies:", cfg["al"]["strategies"], "| shards:", shards)
procs = []
for gpu, shard in enumerate(shards):
    log = WORK / f"al_gpu{gpu}.log"
    stream = log.open("w")
    proc = subprocess.Popen(
        [sys.executable, str(CODE / "scripts" / "06_run_al.py"),
         "--config", str(CFG_KAGGLE), "--data", str(DATA_YAML),
         "--device", str(gpu), "--seed", *map(str, shard)],
        stdout=stream, stderr=subprocess.STDOUT)
    procs.append((gpu, proc, log, stream))

positions = {}
def drain():
    for gpu, _, log, _ in procs:
        with log.open(errors="replace") as reader:
            reader.seek(positions.get(gpu, 0))
            for line in reader:
                print(f"[gpu{gpu}] {line}", end="")
            positions[gpu] = reader.tell()

while any(proc.poll() is None for _, proc, _, _ in procs):
    drain()
    time.sleep(30)
drain()
failed = []
for gpu, proc, log, stream in procs:
    stream.close()
    rc = proc.wait()
    print(f"gpu{gpu} rc={rc} log={log}")
    if rc:
        failed.append((gpu, rc, str(log)))
if failed:
    raise RuntimeError(f"AL subprocess failed: {failed}")

In [ ]:
if RUN_REPORT:
    subprocess.run([sys.executable, str(CODE / "scripts" / "07_al_report.py"),
                    "--config", str(CFG_KAGGLE)], check=True)
report_dir = Path(cfg["reports_dir"]) / "al"
print("reports ->", report_dir)
for path in sorted(report_dir.glob("*")):
    print("  ", path.name)
print("Done. Save Version to keep the output.")

## Resume

Save Version, add the saved output as input to a new session, set `RESUME_DIR` and Run All. A resumed round stops if the code, config, model or dataset differs from the fingerprint stored with it.